In [11]:
from subprocess import check_call

import geopandas as gpd

In [12]:
CPU_PER_PROCESS = 4
OUTPUT_LOCAL = "./output"
OUTPUT_VOLUME = "/usr/src/app/output"
RESOLUTION = 10

In [13]:
TRAIN_PARQUET = f"{OUTPUT_LOCAL}/train_only_biomass.parquet"
TEST_PARQUET = f"{OUTPUT_LOCAL}/test_only_biomass.parquet"
GRIDS = f"{OUTPUT_LOCAL}/tiles.geojson"

In [14]:
grids_df = gpd.read_file(GRIDS)
grids_df

,MEAN_X,MEAN_Y,tile_id,geometry
0,-5.656771,42.484065,028643,"POLYGON ((-5.70677 42.43406, -5.70677 42.53406..."
1,-4.464795,42.661586,038510,"POLYGON ((-4.51479 42.61159, -4.51479 42.71159..."
2,-3.077138,40.386372,045596,"POLYGON ((-3.12714 40.33637, -3.12714 40.43637..."
3,-3.843992,42.718435,043592,"POLYGON ((-3.89399 42.66843, -3.89399 42.76843..."
4,-4.677261,42.357909,036108,"POLYGON ((-4.72726 42.30791, -4.72726 42.40791..."
...,...,...,...,...
173,-1.127566,40.948044,062651,"POLYGON ((-1.17757 40.89804, -1.17757 40.99804..."
174,2.458856,41.898317,093769,"POLYGON ((2.40886 41.84832, 2.40886 41.94832, ..."
175,2.646523,42.228729,095575,"POLYGON ((2.59652 42.17873, 2.59652 42.27873, ..."
176,-4.899820,39.026063,027616,"POLYGON ((-4.94982 38.97606, -4.94982 39.07606..."


In [15]:
train_df = gpd.read_parquet(TRAIN_PARQUET)
train_df

,x,y,year,biomass,tile_id,geometry
0,3137124.7416963745,1810803.9191588927,2020,28.742064,038052,POINT (-3.54712 38.43217)
1,3138084.7416963745,1808543.9191588927,2020,61.379017,038052,POINT (-3.53221 38.41357)
2,3136834.7416963745,1809903.9191588927,2020,41.445705,038052,POINT (-3.54874 38.42368)
3,3137594.7416963745,1810823.9191588927,2020,25.911734,038052,POINT (-3.54186 38.4331)
4,3137914.7416963745,1808923.9191588927,2020,40.892021,038052,POINT (-3.53481 38.41668)
...,...,...,...,...,...,...
5165029,3111664.7416963745,2243933.9191588927,2019,58.634174,035805,POINT (-4.70534 42.23312)
5165030,3111624.7416963745,2243883.9191588927,2019,60.487827,035805,POINT (-4.70571 42.23261)
5165031,3113104.7416963745,2243803.9191588927,2019,118.166924,035805,POINT (-4.68791 42.2345)
5165032,3113074.7416963745,2243833.9191588927,2019,104.021767,035805,POINT (-4.68834 42.23471)


In [16]:
test_df = gpd.read_parquet(TEST_PARQUET)
test_df

,row_id,tile_id,x,y,year,geometry
0,028631_3040064_2251003,028631,3040064.7416963745,2251003.9191588927,2021,POINT (-5.57242 42.1662)
1,028631_3041214_2252063,028631,3041214.7416963745,2252063.9191588927,2021,POINT (-5.56121 42.17768)
2,028631_3041184_2251693,028631,3041184.7416963745,2251693.9191588927,2021,POINT (-5.56071 42.17436)
3,028631_3039864_2250783,028631,3039864.7416963745,2250783.9191588927,2021,POINT (-5.57428 42.16389)
4,028631_3040224_2251483,028631,3040224.7416963745,2251483.9191588927,2021,POINT (-5.57163 42.17073)
...,...,...,...,...,...,...
45014,061751_3374644_2043173,061751,3374644.7416963745,2043173.9191588927,2020,POINT (-1.21909 40.85472)
45015,061751_3374594_2043203,061751,3374594.7416963745,2043203.9191588927,2020,POINT (-1.21972 40.85492)
45016,061751_3374624_2043173,061751,3374624.7416963745,2043173.9191588927,2020,POINT (-1.21932 40.85469)
45017,061751_3374614_2043223,061751,3374614.7416963745,2043223.9191588927,2020,POINT (-1.21952 40.85512)


In [17]:
def run_s2(name: str, roi, sql_where: str = "", dates: tuple[str, str] | None = None):
    start_date, end_date = dates

    print(f"Run S2 {name}")

    cmd = f"""docker container run \
                --name s2_{name} \
                --rm \
                --cpus {CPU_PER_PROCESS} \
                -v {OUTPUT_LOCAL}:{OUTPUT_VOLUME} \
                -e S2_SOURCE=planetary_computer \
                -e S2_START_DATE={start_date} \
                -e S2_END_DATE={end_date} \
                -e S2_ROI_INPUT={roi} \
                -e S2_ROI_SQL_WHERE="{sql_where}" \
                -e S2_RESOLUTION={RESOLUTION} \
                -e S2_BANDS='["B02", "B03", "B04", "B08", "B11", "B12"]' \
                -e S2_OUTPUT_PREFIX={name} \
                eu.gcr.io/ramadhan-s4g/rs-open-source-docker-base:latest \
                .venv/bin/python -m modules.s2_l2a_composite \
    """

    check_call(cmd, shell=True)


In [20]:
def run_grid(index):
    grid = grids_df[index : index + 1]
    xmin, ymin, xmax, ymax = tuple(grid.total_bounds)
    tile_id = grid.iloc[0]["tile_id"]

    print(f"Run tile {tile_id} / {len(grids_df)}")

    train_bbox = train_df.cx[xmin:xmax, ymin:ymax]
    test_bbox = test_df.cx[xmin:xmax, ymin:ymax]

    years = list(set([*train_bbox["year"].unique(), *test_bbox["year"].unique()]))

    for year in years:
        print(f"Run year {year}")

        date_start = f"{year}-06-01"
        date_end = f"{year}-07-31"

        run_s2(
            f"tile_{tile_id}_{year}",
            f"{OUTPUT_VOLUME}/tiles.geojson",
            f"tile_id = '{tile_id}'",
            (date_start, date_end),
        )

In [ ]:
for index in range(1):
    run_grid(index)

Run tile 028643 / 178
Run year 2021
Run S2 tile_028643_2021


2026-08-18 15:27:53 - modules.utils - Load region of interest
2026-08-18 15:27:53 - modules.utils - Process region of interest


0...10...20...30...40...50...60...70...80...90...100 - done.


2026-08-18 15:27:53 - modules.utils - Generate grids
2026-08-18 15:27:53 - modules.utils - Area = 0.009990990990991086 degree squared
2026-08-18 15:27:53 - modules.utils - Area = 12309.900000000118 Ha
Warning 1: Unable to find driver netCDF to unload from GDAL_SKIP environment variable.
2026-08-18 15:27:54 - pyogrio._io - Created 1 records
2026-08-18 15:27:54 - modules.utils - Grids count: 1
2026-08-18 15:27:54 - modules.utils - Process grid Y1_X1


0...10...20...30...40...50...60...70...80...90...100 - done.


2026-08-18 15:27:54 - modules.utils - Generate composite [-5.706726098097289, 42.43410967629796, -5.60681618818738, 42.53410967629796] ('2021-06-01', '2021-07-31')
2026-08-18 15:27:54 - modules.utils - Searching features [-5.706726098097289, 42.43410967629796, -5.60681618818738, 42.53410967629796] ('2021-06-01', '2021-07-31')
2026-08-18 15:27:54 - modules.utils - Found 24 features
2026-08-18 15:27:54 - modules.utils - Generate mask band [-5.706726098097289, 42.43410967629796, -5.60681618818738, 42.53410967629796] ('2021-06-01', '2021-07-31')
